# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [2]:
import torch, re
from PIL import Image
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA", str(Path.cwd().parents[1] / "data")))
manifest_path = prepare_scienceqa(DATA_ROOT / "lab9", smoke=FAST_DEV_RUN)     # та же подвыборка ScienceQA, что в работах 9–10
records = [json.loads(s) for s in manifest_path.read_text().splitlines()]

def rejected_option(r):
    """Следующий по кругу неверный вариант — ошибочный ответ той же формы (одна буква)."""
    k = (r["answer_index"] + 1) % len(r["choices"])
    return chr(65 + k), r["choices"][k]

pairs = []
for r in records:
    if r["split"] not in ("train", "id_test"):
        continue
    rej_letter, rej_text = rejected_option(r)
    pairs.append({"id": r["id"], "split": "train" if r["split"] == "train" else "blind_test", "image": r["image"], "prompt": r["question"],
                  "chosen": r["answer"], "rejected": rej_letter, "chosen_option": r["choices"][r["answer_index"]], "rejected_option": rej_text,
                  "error_type": "wrong_option_adjacent", "image_sha256": r["image_sha256"],
                  "provenance": {"source": r["source_url"], "license": r["license"], "official_split": r["id"].split("-")[1],
                                 "preference_basis": "эталонная разметка ScienceQA (верная буква лучше неверной), а не экспертная оценка полезности"}})
assert all(Path(p["image"]).is_file() and p["chosen"] != p["rejected"] for p in pairs)
print(f"пар: {len(pairs)} (train {sum(p['split'] == 'train' for p in pairs)}, blind_test {sum(p['split'] == 'blind_test' for p in pairs)})")
print(json.dumps(pairs[0], ensure_ascii=False, indent=1)[:700])

пар: 576 (train 512, blind_test 64)
{
 "id": "scienceqa-train-9",
 "split": "train",
 "image": "/Users/filipp/Desktop/рабочий стол/Вуз/учеба/computer_vision/computer_vision_2026/data/lab9/7059b2a9d6213ab2b6bc531c6adaa60811f86552fb2ac3b6acf1244b0485c5b9.png",
 "prompt": "Which of these organisms contains matter that was once part of the phytoplankton?\nA. black rockfish\nB. sea otter\nReturn only the letter of the correct option.",
 "chosen": "A",
 "rejected": "B",
 "chosen_option": "black rockfish",
 "rejected_option": "sea otter",
 "error_type": "wrong_option_adjacent",
 "image_sha256": "7059b2a9d6213ab2b6bc531c6adaa60811f86552fb2ac3b6acf1244b0485c5b9",
 "provenance": {
  "source": "https://huggingface.co/datasets/derek-thom


### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [3]:
from collections import Counter
train_pairs = [p for p in pairs if p["split"] == "train"]; test_pairs = [p for p in pairs if p["split"] == "blind_test"]
assert not {p["id"] for p in train_pairs} & {p["id"] for p in test_pairs}
assert not {p["image_sha256"] for p in train_pairs} & {p["image_sha256"] for p in test_pairs}
with open(ARTIFACTS / "preferences.jsonl", "w") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
audit = {"n_train": len(train_pairs), "n_blind_test": len(test_pairs), "overlap_ids": 0, "overlap_image_hashes": 0,
         "error_types": dict(Counter(p["error_type"] for p in pairs)),
         "chosen_longer_share": float(np.mean([len(p["chosen"]) > len(p["rejected"]) for p in train_pairs])),
         "chosen_shorter_share": float(np.mean([len(p["chosen"]) < len(p["rejected"]) for p in train_pairs])),
         "chosen_letter_distribution": dict(Counter(p["chosen"] for p in train_pairs)),
         "rejected_letter_distribution": dict(Counter(p["rejected"] for p in train_pairs)),
         "pair_letter_patterns_top": Counter(f"{p['chosen']}>{p['rejected']}" for p in train_pairs).most_common(6),
         "note": ("chosen и rejected — одиночные буквы одинаковой длины, длина не может быть скрытым признаком правильности. "
                  "Но rejected всегда «следующий вариант», а A чаще верная буква — возможен позиционный shortcut, см. бонус.")}
save_json("pair_audit.json", audit)
print(json.dumps(audit, ensure_ascii=False, indent=1))

{
 "n_train": 512,
 "n_blind_test": 64,
 "overlap_ids": 0,
 "overlap_image_hashes": 0,
 "error_types": {
  "wrong_option_adjacent": 576
 },
 "chosen_longer_share": 0.0,
 "chosen_shorter_share": 0.0,
 "chosen_letter_distribution": {
  "A": 209,
  "C": 72,
  "B": 230,
  "D": 1
 },
 "rejected_letter_distribution": {
  "B": 209,
  "A": 232,
  "C": 68,
  "D": 3
 },
 "pair_letter_patterns_top": [
  [
   "A>B",
   209
  ],
  [
   "B>A",
   162
  ],
  [
   "C>A",
   69
  ],
  [
   "B>C",
   68
  ],
  [
   "C>D",
   3
  ],
  [
   "D>A",
   1
  ]
 ],
 "note": "chosen и rejected — одиночные буквы одинаковой длины, длина не может быть скрытым признаком правильности. Но rejected всегда «следующий вариант», а A чаще верная буква — возможен позиционный shortcut, см. бонус."
}


### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [4]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
assert DEVICE != "cpu", "DPO VLM требует GPU (CUDA или MPS)"
MODEL_ID, MAX_PIXELS = "Qwen/Qwen2.5-VL-3B-Instruct", 401408
SFT_ADAPTER = Path("../lab_09_llamafactory_vlm_sft/artifacts/lab9_adapter")
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={"": DEVICE})
use_sft = (SFT_ADAPTER / "adapter_model.safetensors").exists()
if use_sft:
    model = PeftModel.from_pretrained(base, str(SFT_ADAPTER), adapter_name="reference", is_trainable=False)
    model.load_adapter(str(SFT_ADAPTER), adapter_name="policy", is_trainable=True)
    source = f"адаптер работы 09 ({SFT_ADAPTER}) — reference и стартовая политика совпадают"
else:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.0)
    model = get_peft_model(base, cfg, adapter_name="policy")
    source = "адаптера работы 09 нет: обе политики начинаются с instruction-tuned модели, это не продолжение собственного SFT"
model.set_adapter("policy")
for n, p in model.named_parameters():
    p.requires_grad = ".policy." in n and "lora" in n
    if p.requires_grad:
        p.data = p.data.float()
trainable = [p for n, p in model.named_parameters() if p.requires_grad]
ref_params = [p for n, p in model.named_parameters() if ".reference." in n]
assert trainable and all(not p.requires_grad for p in ref_params)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
reference_policy = {"source": source, "uses_lab09_sft": use_sft, "base_model": MODEL_ID, "quantization": "bnb nf4, compute fp16",
                    "trainable_params": int(sum(p.numel() for p in trainable)), "reference_trainable_params": int(sum(p.requires_grad for p in ref_params))}
save_json("reference_policy.json", reference_policy)
print(json.dumps(reference_policy, ensure_ascii=False, indent=1))

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{
 "source": "адаптер работы 09 (../lab_09_llamafactory_vlm_sft/artifacts/lab9_adapter) — reference и стартовая политика совпадают",
 "uses_lab09_sft": true,
 "base_model": "Qwen/Qwen2.5-VL-3B-Instruct",
 "quantization": "bnb nf4, compute fp16",
 "trainable_params": 14966784,
 "reference_trainable_params": 0
}


### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [5]:
import torch.nn.functional as F
END = "<|im_end|>"

def prepare_answer(record, answers):
    """Батч: один и тот же промпт с изображением + разные ответы; возвращает входы и длину промпта в токенах."""
    image = Image.open(record["image"]).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": record["prompt"]}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = processor(text=[prompt], images=[image], return_tensors="pt")["input_ids"]
    full = processor(text=[prompt + a + END for a in answers], images=[image] * len(answers), return_tensors="pt", padding=True)
    n_prompt = prompt_ids.shape[1]
    assert torch.equal(full["input_ids"][0, :n_prompt], prompt_ids[0]), "префикс full должен совпадать с отдельно токенизированным prompt"
    return full, n_prompt

def completion_logprob(net, full, n_prompt):
    """Σ log p(токены ответа и <|im_end|>) — prompt (включая токены изображения) в сумму не входит."""
    full = {k: v.to(DEVICE) for k, v in full.items()}
    logits = net(**full).logits.float()
    logp = F.log_softmax(logits[:, :-1], -1)                 # позиция i предсказывает токен i+1
    target = full["input_ids"][:, 1:]
    tok = logp.gather(-1, target[..., None])[..., 0]
    mask = torch.zeros_like(tok); mask[:, n_prompt - 1:] = 1
    mask = mask * full["attention_mask"][:, 1:]
    return (tok * mask).sum(1)

full0, n0 = prepare_answer(train_pairs[0], [train_pairs[0]["chosen"], train_pairs[0]["rejected"]])
n_answer = int(full0["attention_mask"][0].sum()) - n0
print(f"prompt {n0} токенов (вместе с изображением), ответ+конец {n_answer} токена:", processor.tokenizer.convert_ids_to_tokens(full0["input_ids"][0, n0:].tolist()))
lp = completion_logprob(model, full0, n0)
assert lp.requires_grad, "score должен быть дифференцируем по политике"
print("log p(chosen), log p(rejected) у стартовой политики:", lp.detach().cpu().tolist())

prompt 545 токенов (вместе с изображением), ответ+конец 2 токена: ['A', '<|im_end|>']


W1025 16:05:28.882000 81941 torch/_dynamo/convert_frame.py:2008] [1/8] torch._dynamo hit config.recompile_limit (8)
W1025 16:05:28.882000 81941 torch/_dynamo/convert_frame.py:2008] [1/8]    function: '_dequantize_4bit_compute' (/Users/filipp/Desktop/рабочий стол/Вуз/учеба/computer_vision/computer_vision_2026/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:262)
W1025 16:05:28.882000 81941 torch/_dynamo/convert_frame.py:2008] [1/8]    last reason: 1/6: shape[0] == 2048                                         # n = prod(shape)  # bitsandbytes/backends/default/ops.py:271 in _dequantize_4bit_compute
W1025 16:05:28.882000 81941 torch/_dynamo/convert_frame.py:2008] [1/8] User stack trace:
W1025 16:05:28.882000 81941 torch/_dynamo/convert_frame.py:2008] [1/8]   File "/Users/filipp/Desktop/рабочий стол/Вуз/учеба/computer_vision/computer_vision_2026/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py", line 271, in _dequantize_4bit_compute
W1025 16:05:

log p(chosen), log p(rejected) у стартовой политики: [-0.005171974189579487, -5.661296367645264]


### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [6]:
BETA, LR, ACCUM, STEPS = 0.1, 5e-6, 8, (2 if FAST_DEV_RUN else 40)

def dpo_loss(pol_c, pol_r, ref_c, ref_r, beta=BETA):
    margin = (pol_c - ref_c) - (pol_r - ref_r)
    return -F.logsigmoid(beta * margin), margin

optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
z = torch.zeros(1)
l_small, _ = dpo_loss(z + 1.0, z, z, z); l_big, _ = dpo_loss(z + 5.0, z, z, z)
assert l_big < l_small, "больший policy-margin при фиксированном reference должен уменьшать loss"
protocol = {"beta": BETA, "optimizer": "AdamW", "lr": LR, "grad_accumulation": ACCUM, "optimizer_steps": STEPS, "batch_pairs": 1,
            "pairs_used": STEPS * ACCUM, "reference": "тот же адаптер, заморожен, считается под torch.no_grad()", "gradient_checkpointing": True,
            "loss": "-log σ(β[(log π_θ(y_w) − log π_ref(y_w)) − (log π_θ(y_l) − log π_ref(y_l))])"}
save_json("dpo_protocol.json", protocol)
print(json.dumps(protocol, ensure_ascii=False, indent=1))

{
 "beta": 0.1,
 "optimizer": "AdamW",
 "lr": 5e-06,
 "grad_accumulation": 8,
 "optimizer_steps": 40,
 "batch_pairs": 1,
 "pairs_used": 320,
 "reference": "тот же адаптер, заморожен, считается под torch.no_grad()",
 "gradient_checkpointing": true,
 "loss": "-log σ(β[(log π_θ(y_w) − log π_ref(y_w)) − (log π_θ(y_l) − log π_ref(y_l))])"
}


### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [7]:
@torch.no_grad()
def reference_logprob(full, n_prompt):
    model.set_adapter("reference")
    out = completion_logprob(model, full, n_prompt)
    model.set_adapter("policy")
    for p in trainable:
        p.requires_grad = True        # set_adapter может сбросить флаги — возвращаю обучаемость политики
    return out

out_dir = ARTIFACTS / ("aligned_adapter_smoke" if FAST_DEV_RUN else "aligned_adapter") / "train"
saved = out_dir / "policy" / "adapter_model.safetensors"
if saved.exists() and (ARTIFACTS / "dpo_metrics.json").exists():
    # обучение уже выполнено: загружаю сохранённые веса политики, чтобы ответы и слепая оценка остались согласованными
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    res = set_peft_model_state_dict(model, load_file(str(saved)), adapter_name="policy")
    assert not getattr(res, "unexpected_keys", []), res.unexpected_keys
    dpo_saved = json.loads((ARTIFACTS / "dpo_metrics.json").read_text())
    history, k = dpo_saved["history"], dpo_saved["pairs_seen"]
    for h in history:
        if h["step"] % 5 == 0 or h["step"] == 1:
            print(f"шаг {h['step']}/{STEPS}: loss {h['loss']:.4f}, margin {h['margin']:.3f}, accuracy {h['acc']:.2f}, {h['seconds']:.0f} c")
    pol_w = [p for n, p in model.named_parameters() if ".policy." in n]; ref_w = [p for n, p in model.named_parameters() if ".reference." in n]
    with torch.no_grad():
        weight_diff = sum(float((a - b).abs().sum()) for a, b in zip(pol_w, ref_w))
    assert weight_diff > 0, "веса политики должны отличаться от reference"
    model.eval()
    print(f"обучение DPO выполнено в предыдущем запуске ({dpo_saved['train_seconds'] / 60:.0f} мин): загружены сохранённые веса политики")
else:
    order = np.random.RandomState(SEED).permutation(len(train_pairs))
    history, k, t0 = [], 0, time.time()
    model.train()
    for step in range(STEPS):
        optimizer.zero_grad(set_to_none=True)
        rec = {"loss": 0.0, "margin": 0.0, "acc": 0.0}
        for _ in range(ACCUM):
            p = train_pairs[order[k % len(order)]]; k += 1
            full, n_prompt = prepare_answer(p, [p["chosen"], p["rejected"]])
            ref = reference_logprob(full, n_prompt)
            pol = completion_logprob(model, full, n_prompt)
            loss, margin = dpo_loss(pol[0], pol[1], ref[0], ref[1])
            if not torch.isfinite(loss):
                raise FloatingPointError(f"нечисловой DPO loss на шаге {step}: {loss.item()}")
            (loss / ACCUM).backward()
            rec["loss"] += loss.item() / ACCUM; rec["margin"] += margin.item() / ACCUM; rec["acc"] += float(margin.item() > 0) / ACCUM
        gn = torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        history.append({"step": step + 1, **rec, "grad_norm": float(gn), "seconds": time.time() - t0})
        if (step + 1) % 5 == 0 or step == 0:
            print(f"шаг {step + 1}/{STEPS}: loss {rec['loss']:.4f}, margin {rec['margin']:.3f}, accuracy {rec['acc']:.2f}, {time.time() - t0:.0f} c")
    model.eval()
    model.save_pretrained(str(out_dir), selected_adapters=["policy"])
    save_json("dpo_metrics.json", {"history": history, "train_seconds": time.time() - t0, "pairs_seen": k, "steps": len(history),
                                   "final_loss_mean_last5": float(np.mean([h["loss"] for h in history[-5:]]))})
assert len(history) == STEPS and all(np.isfinite(h["loss"]) for h in history)
print("адаптер сохранён:", sorted(p.name for p in out_dir.rglob("*") if p.is_file())[:5])

шаг 1/40: loss 0.6939, margin -0.016, accuracy 0.38, 73 c
шаг 5/40: loss 0.6840, margin 0.184, accuracy 1.00, 362 c
шаг 10/40: loss 0.6726, margin 0.416, accuracy 1.00, 769 c
шаг 15/40: loss 0.6634, margin 0.604, accuracy 1.00, 1151 c
шаг 20/40: loss 0.6561, margin 0.760, accuracy 1.00, 1547 c
шаг 25/40: loss 0.6471, margin 0.950, accuracy 0.88, 1904 c
шаг 30/40: loss 0.6436, margin 1.027, accuracy 0.88, 2254 c
шаг 35/40: loss 0.5917, margin 2.149, accuracy 1.00, 2621 c
шаг 40/40: loss 0.6130, margin 1.702, accuracy 0.88, 2982 c
обучение DPO выполнено в предыдущем запуске (50 мин): загружены сохранённые веса политики
адаптер сохранён: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']


### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


In [8]:
@torch.no_grad()
def answer(net_mode, record):
    """Детерминированный ответ: foundation — адаптеры отключены, reference/aligned — соответствующий адаптер."""
    image = Image.open(record["image"]).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": record["prompt"]}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)
    if net_mode == "foundation":
        with model.disable_adapter():
            out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    else:
        model.set_adapter("policy" if net_mode == "aligned" else "reference")
        out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    return processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

blind_path = ARTIFACTS / "blind_predictions.jsonl"
cached = {json.loads(l)["id"]: json.loads(l) for l in blind_path.read_text().splitlines()} if blind_path.exists() else {}
use_cached = set(cached) == {p["id"] for p in test_pairs}
if use_cached:     # ответы уже получены в предыдущем запуске — по ним заполнена слепая оценка, перегенерация её бы разошлась
    base_answers = {i: {"foundation": c["foundation"], "reference": c["reference"]} for i, c in cached.items()}
    print("ответы на blind test взяты из сохранённого blind_predictions.jsonl")
else:
    base_answers = {p["id"]: {"foundation": answer("foundation", p), "reference": answer("reference", p)} for p in test_pairs}
same = np.mean([v["foundation"] == v["reference"] for v in base_answers.values()])
print(f"ответов на blind test: {len(base_answers)}; reference — SFT работы 09: {use_sft}; совпадение foundation и reference: {same:.2f}")

ответы на blind test взяты из сохранённого blind_predictions.jsonl
ответов на blind test: 64; reference — SFT работы 09: True; совпадение foundation и reference: 0.78


### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [9]:
if not use_cached:
    with open(blind_path, "w") as f:
        for p in test_pairs:
            f.write(json.dumps({"id": p["id"], "reference_letter": p["chosen"], "foundation": base_answers[p["id"]]["foundation"],
                                "reference": base_answers[p["id"]]["reference"], "aligned": answer("aligned", p)}, ensure_ascii=False) + "\n")
blind = [json.loads(l) for l in blind_path.read_text().splitlines()]
assert {b["id"] for b in blind} == {p["id"] for p in test_pairs} and all(b["aligned"] is not None for b in blind)
print("пример:", blind[0])

пример: {'id': 'scienceqa-test-10', 'reference_letter': 'C', 'foundation': 'C', 'reference': 'C', 'aligned': 'C'}


### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [10]:
def em(ans, ref):
    return float(ans.strip() == ref)
objective = {pol: {"n": len(blind), "exact_match": float(np.mean([em(b[pol], b["reference_letter"]) for b in blind])),
                   "format_single_letter": float(np.mean([bool(re.fullmatch(r"[A-Z]", b[pol].strip())) for b in blind]))}
             for pol in ("foundation", "reference", "aligned")}
objective["note"] = "EM по эталонной букве — автоматическая метрика, а не человеческий win rate"
save_json("objective_scores.json", objective)
print(json.dumps(objective, ensure_ascii=False, indent=1))

# Слепая форма: в каждой записи три ответа в случайном порядке под метками X/Y/Z, ключ хранится отдельно
judge_path = ARTIFACTS / "blind_judgements.jsonl"
existing = {json.loads(l)["id"]: json.loads(l) for l in judge_path.read_text().splitlines()} if judge_path.exists() else {}
rs = np.random.RandomState(SEED + 11)
rows, key = [], {}
recs = {p["id"]: p for p in test_pairs}
for b in blind:
    perm = rs.permutation(["foundation", "reference", "aligned"]).tolist()
    key[b["id"]] = dict(zip("XYZ", perm))
    old = existing.get(b["id"], {})
    rows.append({"id": b["id"], "question": recs[b["id"]]["prompt"], "image": recs[b["id"]]["image"],
                 "answers": {s: b[pol] for s, pol in zip("XYZ", perm)},
                 "best": old.get("best", "pending"), "hallucination": old.get("hallucination", {s: "pending" for s in "XYZ"}),
                 "format_ok": old.get("format_ok", {s: "pending" for s in "XYZ"}), "notes": old.get("notes", "")})
judge_path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))
(ARTIFACTS / "blind_key.json").write_text(json.dumps(key, indent=1))
print(f"форма ручной оценки: {len(rows)} записей, заполнено {sum(r['best'] != 'pending' for r in rows)}; "
      "best — метка лучшего ответа, несколько равно лучших ('XY') или 'tie'")

{
 "foundation": {
  "n": 64,
  "exact_match": 0.828125,
  "format_single_letter": 0.90625
 },
 "reference": {
  "n": 64,
  "exact_match": 0.890625,
  "format_single_letter": 1.0
 },
 "aligned": {
  "n": 64,
  "exact_match": 0.90625,
  "format_single_letter": 1.0
 },
 "note": "EM по эталонной букве — автоматическая метрика, а не человеческий win rate"
}
форма ручной оценки: 64 записей, заполнено 64; best — метка лучшего ответа, несколько равно лучших ('XY') или 'tie'


### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [11]:
judged = [json.loads(l) for l in judge_path.read_text().splitlines()]
done = [j for j in judged if j["best"] != "pending" and "pending" not in j["hallucination"].values() and "pending" not in j["format_ok"].values()]
report = {"objective_em": {k: v["exact_match"] for k, v in objective.items() if k != "note"}, "n_judged": len(done), "n_total": len(judged),
          "complete": len(done) == len(judged)}
if done:
    wins = {"aligned_vs_reference": [], "aligned_vs_foundation": []}
    hall = {pol: [] for pol in ("foundation", "reference", "aligned")}; fmt = {pol: [] for pol in hall}
    for j in done:
        inv = {pol: s for s, pol in key[j["id"]].items()}
        best = set("XYZ") if j["best"] == "tie" else set(j["best"])   # 'X', несколько равно лучших ('XY') или 'tie'
        for pol in hall:
            hall[pol].append(bool(j["hallucination"][inv[pol]])); fmt[pol].append(bool(j["format_ok"][inv[pol]]))
        for name, other in (("aligned_vs_reference", "reference"), ("aligned_vs_foundation", "foundation")):
            a_best, o_best = inv["aligned"] in best, inv[other] in best
            wins[name].append(1.0 if a_best and not o_best else 0.0 if o_best and not a_best else 0.5)
    report.update({"win_rate": {k: float(np.mean(v)) for k, v in wins.items()},
                   "hallucination_rate": {k: float(np.mean(v)) for k, v in hall.items()},
                   "format_compliance": {k: float(np.mean(v)) for k, v in fmt.items()}})
report["limitations"] = ("предпочтения выведены из эталонных букв ScienceQA: улучшение означает лишь, что модель чаще выбирает верный "
                         "вариант на этом распределении, и не говорит о безопасности или общей полезности ответов")
save_json("blind_report.json", report)
print(json.dumps(report, ensure_ascii=False, indent=1))

{
 "objective_em": {
  "foundation": 0.828125,
  "reference": 0.890625,
  "aligned": 0.90625
 },
 "n_judged": 64,
 "n_total": 64,
 "complete": true,
 "win_rate": {
  "aligned_vs_reference": 0.5078125,
  "aligned_vs_foundation": 0.5625
 },
 "hallucination_rate": {
  "foundation": 0.015625,
  "reference": 0.0,
  "aligned": 0.0
 },
 "format_compliance": {
  "foundation": 0.90625,
  "reference": 1.0,
  "aligned": 1.0
 },
 "limitations": "предпочтения выведены из эталонных букв ScienceQA: улучшение означает лишь, что модель чаще выбирает верный вариант на этом распределении, и не говорит о безопасности или общей полезности ответов"
}


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [12]:
# Гипотеза (до просмотра результатов): rejected в парах всегда «следующий по кругу» вариант, поэтому DPO сдвигает
# не столько знание, сколько позиционное поведение: aligned реже выбирает букву, стоящую сразу после верной,
# и распределение выбранных букв меняется. Проверка — на validation (64 примера), который не участвовал ни в DPO,
# ни в blind test. Правило решения задано заранее: гипотеза подтверждается, если доля ответов «следующая за верной»
# у aligned ниже, чем у reference, и TV-расстояние между распределениями букв > 0,05.
val = [{"id": r["id"], "image": r["image"], "prompt": r["question"], "chosen": r["answer"], "n_choices": len(r["choices"])}
       for r in records if r["split"] == "validation"]
prev = json.loads((ARTIFACTS / "bonus_results.json").read_text()) if (ARTIFACTS / "bonus_results.json").exists() else None
if prev and all(len(prev["answers"].get(pol, [])) == len(val) for pol in ("reference", "aligned")):
    val_ans = prev["answers"]; print("ответы validation взяты из сохранённого bonus_results.json")
else:
    val_ans = {pol: [answer(pol, v) for v in val] for pol in ("reference", "aligned")}

def letter_stats(ans):
    nxt = [a.strip() == chr(65 + (ord(v["chosen"]) - 65 + 1) % v["n_choices"]) for a, v in zip(ans, val)]
    dist = Counter(a.strip()[:1] for a in ans)
    return {"accuracy": float(np.mean([a.strip() == v["chosen"] for a, v in zip(ans, val)])), "next_letter_share": float(np.mean(nxt)),
            "letter_distribution": {k: v / len(ans) for k, v in sorted(dist.items())}}

st = {pol: letter_stats(a) for pol, a in val_ans.items()}
letters = sorted(set(st["reference"]["letter_distribution"]) | set(st["aligned"]["letter_distribution"]))
tv = 0.5 * sum(abs(st["reference"]["letter_distribution"].get(l, 0) - st["aligned"]["letter_distribution"].get(l, 0)) for l in letters)
confirmed = st["aligned"]["next_letter_share"] < st["reference"]["next_letter_share"] and tv > 0.05
bonus_md = f"""# Мини-исследование: позиционный сдвиг после DPO

**Гипотеза.** Rejected-ответ в каждой паре — следующий по кругу вариант, поэтому DPO может менять позиционное поведение
(реже выбирать «следующую за верной» букву) сильнее, чем знание. Проверка на validation ScienceQA (n={len(val)}), который
не использовался ни в обучении DPO, ни в blind test.

**Правило (задано до просмотра):** подтверждено, если доля ответов «следующая за верной буква» у aligned ниже, чем у reference,
и TV-расстояние между распределениями выбранных букв больше 0,05.

| политика | accuracy | доля «следующей за верной» | распределение букв |
|---|---:|---:|---|
| reference (SFT) | {st['reference']['accuracy']:.3f} | {st['reference']['next_letter_share']:.3f} | {st['reference']['letter_distribution']} |
| aligned (DPO) | {st['aligned']['accuracy']:.3f} | {st['aligned']['next_letter_share']:.3f} | {st['aligned']['letter_distribution']} |

TV-расстояние распределений букв: {tv:.3f}. **Итог по правилу: гипотеза {'подтверждена' if confirmed else 'не подтверждена'}.**
Сравнение с основным экспериментом и интерпретация — в README работы.
"""
(ARTIFACTS / "bonus.md").write_text(bonus_md)
save_json("bonus_results.json", {"stats": st, "tv_distance": tv, "confirmed": confirmed, "answers": val_ans})
print(bonus_md)

ответы validation взяты из сохранённого bonus_results.json
# Мини-исследование: позиционный сдвиг после DPO

**Гипотеза.** Rejected-ответ в каждой паре — следующий по кругу вариант, поэтому DPO может менять позиционное поведение
(реже выбирать «следующую за верной» букву) сильнее, чем знание. Проверка на validation ScienceQA (n=64), который
не использовался ни в обучении DPO, ни в blind test.

**Правило (задано до просмотра):** подтверждено, если доля ответов «следующая за верной буква» у aligned ниже, чем у reference,
и TV-расстояние между распределениями выбранных букв больше 0,05.

| политика | accuracy | доля «следующей за верной» | распределение букв |
|---|---:|---:|---|
| reference (SFT) | 0.844 | 0.125 | {'A': 0.390625, 'B': 0.515625, 'C': 0.09375} |
| aligned (DPO) | 0.859 | 0.125 | {'A': 0.40625, 'B': 0.515625, 'C': 0.078125} |

TV-расстояние распределений букв: 0.016. **Итог по правилу: гипотеза не подтверждена.**
Сравнение с основным экспериментом и интерпретация — в README

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.